# 02 - Hooks and Exit Conditions (Verbose Trace)

This notebook extends the base loop with hook points that can shape behavior at runtime.

Reading order:
1. Setup and adapter config.
2. Domain tools and world state.
3. Hook policy class (`session_start`, `pre_tool_use`, `post_tool_use`, `stop`).
4. Shared trace + tool execution helpers.
5. Main loop that integrates hooks on every iteration.
6. Demo run and trace inspection.

Why it matters:
- Hooks let you enforce constraints without changing core adapter code.
- Stop hooks are useful when the model tries to end too early.


## Step 1 - Environment and Adapter Setup

Centralized model/trace configuration for the hook-driven loop.


In [ ]:
# Section: Setup
# Purpose: Import dependencies, configure tracing defaults, and initialize the adapter.

import json
import random
import re
import sys
import time
from pathlib import Path
from typing import Any

# Ensure repo root is importable when running this notebook from its folder.
repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "orchestrator").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from orchestrator.llm_interaction.adapter import LLMAdapter, LLMError

MODEL = "gpt-oss:20b"
MAX_ITERATIONS = 60
TRACE_PREVIEW_CHARS = 500

# Maximum-detail tracing controls
FULL_TRACE = True
SHOW_THINKING_TRACE = True
SHOW_RAW_RESPONSE = True
SHOW_RESPONSE_STATS = True
PRINT_FULL_TRACE_AFTER_RUN = True
PRINT_FULL_RESPONSE_SNAPSHOTS_AFTER_RUN = True

adapter = LLMAdapter(model=MODEL, verbose=False)


## Step 2 - Domain State and Tools

Defines a compact world state and a small tool surface for experiments.


In [ ]:
# Section: Domain state + tools
# Purpose: Define world data and tool functions used by the hook-enabled orchestration loop.

world_state = {
    "location": "Moonfall Keep",
    "time": "20:00",
    "threat_level": "medium",
}


def get_world_state() -> dict[str, Any]:
    return world_state


def advance_clock(hours: int) -> dict[str, Any]:
    h, m = map(int, world_state["time"].split(":"))
    h = (h + int(hours)) % 24
    world_state["time"] = f"{h:02d}:{m:02d}"
    return world_state


def roll_check(dc: int, modifier: int = 0) -> dict[str, Any]:
    roll = random.randint(1, 20)
    total = roll + int(modifier)
    return {
        "roll": roll,
        "modifier": int(modifier),
        "total": total,
        "dc": int(dc),
        "success": total >= int(dc),
    }


TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_world_state",
            "description": "Read current world state.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "advance_clock",
            "description": "Advance world clock by N hours.",
            "parameters": {
                "type": "object",
                "properties": {"hours": {"type": "integer", "minimum": 1, "maximum": 24}},
                "required": ["hours"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "roll_check",
            "description": "Roll 1d20 plus modifier against a DC.",
            "parameters": {
                "type": "object",
                "properties": {
                    "dc": {"type": "integer"},
                    "modifier": {"type": "integer", "default": 0},
                },
                "required": ["dc"],
            },
        },
    },
]

TOOL_IMPL = {
    "get_world_state": get_world_state,
    "advance_clock": advance_clock,
    "roll_check": roll_check,
}


## Step 3 - Hook Policy Layer

`DMHooks` is the policy layer:
- add startup context
- block risky tool calls
- inject recovery hints after failures
- enforce completion rules


In [ ]:
# Section: Hook policy
# Purpose: Encapsulate session/tool/stop control rules that steer loop behavior.

class DMHooks:
    def session_start(self, state: dict[str, Any]) -> str | None:
        return (
            "Hook context: keep continuity with prior events and keep danger proportional to threat_level. "
            f"Current threat_level is {state.get('threat_level')}."
        )

    def pre_tool_use(self, tool_name: str, args: dict[str, Any], state: dict[str, Any]) -> dict[str, Any]:
        if tool_name == "advance_clock" and int(args.get("hours", 0)) > 8:
            return {"allow": False, "reason": "Do not skip more than 8 hours in a single turn."}
        return {"allow": True}

    def post_tool_use(self, tool_name: str, tool_result: dict[str, Any], state: dict[str, Any]) -> str | None:
        if not tool_result.get("ok", True):
            return "A tool failed. Recover by explaining the constraint and offering alternatives."
        return None

    def stop(self, assistant_text: str, state: dict[str, Any], stop_hook_active: bool) -> str | None:
        text = (assistant_text or "").lower()
        has_choices = ("what do you do" in text) or ("choice" in text) or ("options" in text)
        if not has_choices:
            return "Before completing, provide 2-3 concrete player choices."

        if stop_hook_active and len(assistant_text.strip()) < 80:
            return "Your response is too brief. Provide richer scene detail before ending."

        return None


hooks = DMHooks()


## Step 4 - Trace and Execution Helpers

Reusable logging + parsing helpers keep the main loop readable.


In [ ]:
# Section: Runtime helpers
# Purpose: Provide consistent tracing, response parsing, and safe tool execution.

RUN_STATE = {
    "total_tool_calls": 0,
}


def reset_run_state() -> None:
    RUN_STATE["total_tool_calls"] = 0


def ts() -> str:
    return time.strftime("%H:%M:%S")


def preview(text: str, limit: int = TRACE_PREVIEW_CHARS) -> str:
    if text is None:
        return ""
    text = str(text).strip()
    if FULL_TRACE:
        return text
    if len(text) <= limit:
        return text
    return text[:limit] + f" ... [truncated {len(text) - limit} chars]"


def trace_print(tag: str, message: str, trace_lines: list[str] | None = None) -> None:
    line = f"[{ts()}] [{tag}] {message}"
    print(line, flush=True)
    if trace_lines is not None:
        trace_lines.append(line)


def response_to_dict(response: Any) -> dict[str, Any]:
    if hasattr(response, "model_dump"):
        try:
            data = response.model_dump(exclude_none=False)
            if isinstance(data, dict):
                return data
        except Exception:
            pass

    if isinstance(response, dict):
        return response

    return {
        "_type": type(response).__name__,
        "repr": repr(response),
    }


def extract_message_dict(response: Any) -> dict[str, Any]:
    data = response_to_dict(response)
    msg = data.get("message") if isinstance(data, dict) else None
    if isinstance(msg, dict):
        return msg
    if hasattr(msg, "model_dump"):
        try:
            dumped = msg.model_dump(exclude_none=False)
            if isinstance(dumped, dict):
                return dumped
        except Exception:
            pass
    return {}


def extract_thinking_text(response: Any) -> str:
    msg = extract_message_dict(response)
    thinking = msg.get("thinking", "")

    if isinstance(thinking, list):
        return "".join(str(x) for x in thinking)

    return str(thinking or "")


def normalize_tool_calls(response: Any) -> list[dict[str, Any]]:
    out: list[dict[str, Any]] = []
    raw_calls = adapter._extract_tool_calls(response)

    for i, call in enumerate(raw_calls):
        if not isinstance(call, dict):
            continue

        fn = call.get("function", {})
        name = fn.get("name") or call.get("name")
        args = fn.get("arguments", {})

        if isinstance(args, str):
            try:
                args = json.loads(args)
            except json.JSONDecodeError:
                args = {}

        if not isinstance(args, dict):
            args = {}

        if not name:
            continue

        out.append(
            {
                "id": call.get("id") or f"call_{i}",
                "name": str(name),
                "arguments": args,
                "raw": call,
                "source": "structured",
            }
        )

    return out


def extract_fallback_tool_calls_from_text(text: str) -> list[dict[str, Any]]:
    if not text:
        return []

    calls: list[dict[str, Any]] = []

    for line in text.splitlines():
        stripped = line.strip().rstrip(",")
        if not stripped.startswith("{") or not stripped.endswith("}"):
            continue
        try:
            obj = json.loads(stripped)
        except json.JSONDecodeError:
            continue
        if not isinstance(obj, dict):
            continue
        name = obj.get("name")
        params = obj.get("parameters", {})
        if not name or not isinstance(params, dict):
            continue
        calls.append(
            {
                "id": f"text_line_{len(calls)}",
                "name": str(name),
                "arguments": params,
                "raw": {
                    "function": {
                        "name": str(name),
                        "arguments": params,
                    }
                },
                "source": "text_fallback",
            }
        )

    pattern = re.compile(
        r'\{\s*"name"\s*:\s*"(?P<name>[^"]+)"\s*,\s*"parameters"\s*:\s*(?P<params>\{.*?\})\s*\}',
        flags=re.DOTALL,
    )

    for match in pattern.finditer(text):
        name = match.group("name")
        params_raw = match.group("params")
        try:
            params = json.loads(params_raw)
        except json.JSONDecodeError:
            continue
        if not isinstance(params, dict):
            continue

        duplicate = any(c["name"] == str(name) and c["arguments"] == params for c in calls)
        if duplicate:
            continue

        calls.append(
            {
                "id": f"text_regex_{len(calls)}",
                "name": str(name),
                "arguments": params,
                "raw": {
                    "function": {
                        "name": str(name),
                        "arguments": params,
                    }
                },
                "source": "text_fallback",
            }
        )

    return calls


def execute_tool(name: str, arguments: dict[str, Any], trace_lines: list[str]) -> dict[str, Any]:
    RUN_STATE["total_tool_calls"] += 1
    call_no = RUN_STATE["total_tool_calls"]

    trace_print("TOOL-CALL", f"#{call_no} {name} args={preview(json.dumps(arguments, ensure_ascii=True))}", trace_lines)

    fn = TOOL_IMPL.get(name)
    if not fn:
        payload = {"ok": False, "error": f"Unknown tool: {name}"}
        trace_print("TOOL-RESULT", f"#{call_no} {name} -> {payload['error']}", trace_lines)
        return payload

    try:
        payload = {"ok": True, "result": fn(**arguments)}
    except Exception as exc:
        payload = {"ok": False, "error": str(exc)}

    trace_print("TOOL-RESULT", f"#{call_no} {name} -> {preview(json.dumps(payload, ensure_ascii=True))}", trace_lines)
    return payload


## Step 5 - Hook-Aware Main Loop

This loop applies hooks at specific points:
- before first model call (`session_start`)
- before each tool (`pre_tool_use`)
- after each tool (`post_tool_use`)
- before completion (`stop`)


In [ ]:
# Section: Main loop with hooks
# Purpose: Orchestrate model/tool rounds while enforcing hook-driven constraints and retries.

SYSTEM_PROMPT = (
    "You are a tactical and cinematic DnD DM. Use tools for world checks and world updates. "
    "Do not fabricate tool results. Emit detailed reasoning in your thinking stream."
)


def run_loop_with_hooks(user_prompt: str, max_iterations: int = MAX_ITERATIONS) -> dict[str, Any]:
    reset_run_state()
    trace_lines: list[str] = []
    response_snapshots: list[dict[str, Any]] = []
    stop_hook_active = False

    messages: list[dict[str, Any]] = []

    start_ctx = hooks.session_start(world_state)
    if start_ctx:
        messages.append({"role": "user", "content": start_ctx})
        trace_print("HOOK-START", start_ctx, trace_lines)

    messages.append({"role": "user", "content": user_prompt})

    trace_print("RUN", f"model={MODEL} max_iterations={max_iterations}", trace_lines)

    for iteration in range(1, max_iterations + 1):
        trace_print("ITER", f"{iteration}/{max_iterations} stop_hook_active={stop_hook_active}", trace_lines)

        try:
            response = adapter.request_with_tools(
                stage="notebook_02_verbose",
                system_prompt=SYSTEM_PROMPT,
                messages=messages,
                tools=TOOLS,
            )
        except LLMError as exc:
            trace_print("MODEL-ERROR", str(exc), trace_lines)
            return {
                "status": "error",
                "error": str(exc),
                "messages": messages,
                "trace_lines": trace_lines,
                "response_snapshots": response_snapshots,
            }

        response_dict = response_to_dict(response)
        response_snapshots.append(response_dict)

        if SHOW_RESPONSE_STATS:
            stats = {
                "model": response_dict.get("model"),
                "done_reason": response_dict.get("done_reason"),
                "prompt_eval_count": response_dict.get("prompt_eval_count"),
                "eval_count": response_dict.get("eval_count"),
                "total_duration": response_dict.get("total_duration"),
            }
            trace_print("MODEL-STATS", json.dumps(stats, ensure_ascii=True), trace_lines)

        if SHOW_RAW_RESPONSE:
            trace_print("RAW-RESPONSE", json.dumps(response_dict, indent=2, ensure_ascii=True), trace_lines)

        assistant_text = adapter._extract_content(response).strip()
        thinking_text = extract_thinking_text(response).strip()

        trace_print("ASSISTANT-CONTENT", preview(assistant_text or "<empty>"), trace_lines)
        if SHOW_THINKING_TRACE:
            trace_print("ASSISTANT-THINKING", preview(thinking_text or "<none>"), trace_lines)

        tool_calls = normalize_tool_calls(response)
        if not tool_calls:
            fallback_source = "\n".join(x for x in [assistant_text, thinking_text] if x)
            fallback_calls = extract_fallback_tool_calls_from_text(fallback_source)
            if fallback_calls:
                tool_calls = fallback_calls
                trace_print("FALLBACK", f"Recovered {len(tool_calls)} text-encoded tool calls.", trace_lines)

        if tool_calls:
            messages.append(
                {
                    "role": "assistant",
                    "content": assistant_text,
                    "tool_calls": [c["raw"] for c in tool_calls],
                }
            )

            for call in tool_calls:
                trace_print("TOOL-SOURCE", f"{call['name']} via {call.get('source', 'unknown')}", trace_lines)
                pre = hooks.pre_tool_use(call["name"], call["arguments"], world_state)
                trace_print("HOOK-PRE", f"{call['name']} -> {json.dumps(pre, ensure_ascii=True)}", trace_lines)

                if not pre.get("allow", True):
                    tool_payload = {"ok": False, "error": pre.get("reason", "Blocked by pre_tool_use")}
                else:
                    tool_payload = execute_tool(call["name"], call["arguments"], trace_lines)

                messages.append(
                    {
                        "role": "tool",
                        "tool_name": call["name"],
                        "content": json.dumps(tool_payload, separators=(",", ":"), ensure_ascii=True),
                    }
                )

                post_ctx = hooks.post_tool_use(call["name"], tool_payload, world_state)
                if post_ctx:
                    trace_print("HOOK-POST", post_ctx, trace_lines)
                    messages.append({"role": "user", "content": f"Hook note: {post_ctx}"})

            trace_print("PROGRESS", f"tool_calls={RUN_STATE['total_tool_calls']}", trace_lines)
            continue

        reason = hooks.stop(assistant_text, world_state, stop_hook_active)
        if reason:
            stop_hook_active = True
            trace_print("HOOK-STOP", reason, trace_lines)
            messages.append({"role": "assistant", "content": assistant_text})
            messages.append(
                {
                    "role": "user",
                    "content": f"You were about to finish, but a stop hook blocked completion: {reason}",
                }
            )
            continue

        messages.append({"role": "assistant", "content": assistant_text})
        return {
            "status": "completed",
            "final_answer": assistant_text,
            "rounds": iteration,
            "messages": messages,
            "trace_lines": trace_lines,
            "response_snapshots": response_snapshots,
            "tool_calls": RUN_STATE["total_tool_calls"],
        }

    return {
        "status": "max_iterations",
        "final_answer": "Stopped due to max iterations.",
        "messages": messages,
        "trace_lines": trace_lines,
        "response_snapshots": response_snapshots,
        "tool_calls": RUN_STATE["total_tool_calls"],
    }


## Step 6 - Demo Run and Inspection

Runs one scenario and prints outcome, trace log, and optional raw snapshots.


In [ ]:
# Section: Demo execution
# Purpose: Run a sample scenario and inspect hook/tool decisions end-to-end.

result = run_loop_with_hooks("We sneak into Moonfall Keep at night and scout the west tower.")

print("status:", result.get("status"))
print("rounds:", result.get("rounds"))
print("tool_calls:", result.get("tool_calls"))
print()
print("FINAL ANSWER")
print(result.get("final_answer", ""))

trace_lines = result.get("trace_lines", [])
print("\nTRACE LINE COUNT:", len(trace_lines))
if PRINT_FULL_TRACE_AFTER_RUN:
    for line in trace_lines:
        print(line)

snapshots = result.get("response_snapshots", [])
print("\nRESPONSE SNAPSHOT COUNT:", len(snapshots))
if PRINT_FULL_RESPONSE_SNAPSHOTS_AFTER_RUN:
    for i, snap in enumerate(snapshots, start=1):
        print(f"\n--- SNAPSHOT {i} ---")
        print(json.dumps(snap, indent=2, ensure_ascii=True))
